# Using Model Outputs in Decision Calculations (SOLUTION)

## Scenario

Ledge & Lend Group has two credit-risk models: a gradient boosting
classifier that predicts the probability of default (`predicted_pd`) and a gradient
boosting regressor that predicts loss given default (`predicted_lgd`). Both outputs
are combined in an expected-value function to choose the profit-maximizing approval
threshold from three candidates: Strict (PD < 15%), Moderate (PD < 25%), Permissive
(PD < 40%).

## What this notebook delivers

1. Provided model-training pipeline (classifier + regressor) for context.
2. `applicant_ev()` using both `predicted_pd` and `predicted_lgd`.
3. Threshold comparison table.
4. Defended recommendation.

## Setup

In [1]:
import numpy as np
import pandas as pd

DATA_PATH = "../using-model-outputs-starter/data/loan_applicants.csv"
HIST_PATH = "../using-model-outputs-starter/data/historical_loans.csv"

LOAN_TERM_YEARS = 3     # 3-year personal loan
RISK_AVERSION   = 0.10  # λ: expected-utility penalty per $1 of portfolio SD
THRESHOLDS = {
    "Strict":     0.15,
    "Moderate":   0.25,
    "Permissive": 0.40,
}

## Provided: How the credit-risk models were built

In [2]:
# ── PROVIDED — predictions are already in loan_applicants.csv ─────────────────
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor

FEATURES = ['credit_score', 'annual_income_usd', 'debt_to_income_ratio',
            'employment_tenure_years']

hist = pd.read_csv(HIST_PATH)

clf = GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                  learning_rate=0.05, random_state=42)
clf.fit(hist[FEATURES], hist['defaulted'])

defaults = hist[hist['defaulted'] == 1]
reg = GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                 learning_rate=0.05, random_state=42)
reg.fit(defaults[FEATURES], defaults['loss_given_default'])

#   applicants['predicted_pd']  = clf.predict_proba(applicants[FEATURES])[:, 1]
#   applicants['predicted_lgd'] = reg.predict(applicants[FEATURES]).clip(0.22, 0.90)
# ─────────────────────────────────────────────────────────────────────────────

,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.05
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",200
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are""friedman_mse"" for the mean squared error with improvement score byFriedman, ""squared_error"" for mean squared error. The default value of""friedman_mse"" is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft

## 1. Load the applicant data

In [3]:
applicants = pd.read_csv(DATA_PATH)
print(f"Shape: {applicants.shape}")
applicants.head()

Shape: (500, 9)


,applicant_id,credit_score,annual_income_usd,debt_to_income_ratio,employment_tenure_years,loan_amount_usd,annual_interest_rate,predicted_pd,predicted_lgd
0,CB-001,792,75694,0.299,1.3,5021,0.0883,0.2148,0.6071
1,CB-002,759,104146,0.294,7.0,24728,0.1035,0.2461,0.6672
2,CB-003,751,30212,0.310,2.8,7181,0.1290,0.3458,0.7991
3,CB-004,811,64162,0.225,0.5,3357,0.0834,0.2625,0.7401
4,CB-005,712,38497,0.357,0.5,4712,0.1370,0.3701,0.9000


## 2. Implement `applicant_ev()`

Both model outputs enter the formula directly:

$$
\text{EV} = \text{amount} \times \text{rate} \times \text{term}
  \times (1 - \text{PD})
\;-\;
\text{amount} \times \text{LGD} \times \text{PD}
$$

`predicted_lgd` replaces the fixed constant used in simpler approaches.
Higher-risk applicants have both elevated PD and elevated LGD, so they are
penalised on both sides of the equation.

In [4]:
def applicant_ev(predicted_pd: float, loan_amount: float, annual_rate: float,
                 predicted_lgd: float,
                 loan_term_years: float = LOAN_TERM_YEARS) -> float:
    """Expected net profit from approving one loan applicant."""
    interest_income = loan_amount * annual_rate * loan_term_years * (1 - predicted_pd)
    expected_loss   = loan_amount * predicted_lgd * predicted_pd
    return interest_income - expected_loss

## 2b. Implement `applicant_variance()`

Under a Bernoulli default model, each applicant has two outcomes: repay (earn
interest) or default (lose LGD × principal). The variance of that distribution
measures how much the actual outcome could differ from the expected profit.
Portfolio variance is the sum of individual variances (independent defaults).

In [5]:
def applicant_variance(predicted_pd: float, loan_amount: float, annual_rate: float,
                       predicted_lgd: float,
                       loan_term_years: float = LOAN_TERM_YEARS) -> float:
    """Variance of profit for one loan applicant (Bernoulli default model)."""
    profit_if_repay =  loan_amount * annual_rate * loan_term_years
    loss_if_default = -loan_amount * predicted_lgd
    ev = applicant_ev(predicted_pd, loan_amount, annual_rate, predicted_lgd, loan_term_years)
    return (profit_if_repay - ev)**2 * (1 - predicted_pd) + (loss_if_default - ev)**2 * predicted_pd

## 3. Apply `applicant_ev()` to the full dataset

In [6]:
applicants["ev_usd"]  = applicant_ev(
    applicants["predicted_pd"],
    applicants["loan_amount_usd"],
    applicants["annual_interest_rate"],
    applicants["predicted_lgd"],
)
applicants["var_usd"] = applicant_variance(
    applicants["predicted_pd"],
    applicants["loan_amount_usd"],
    applicants["annual_interest_rate"],
    applicants["predicted_lgd"],
)
applicants.head()

,applicant_id,credit_score,annual_income_usd,debt_to_income_ratio,employment_tenure_years,loan_amount_usd,annual_interest_rate,predicted_pd,predicted_lgd,ev_usd,var_usd
0,CB-001,792,75694,0.299,1.3,5021,0.0883,0.2148,0.6071,389.601482,3.233166e+06
1,CB-002,759,104146,0.294,7.0,24728,0.1035,0.2461,0.6672,1728.191206,1.084463e+08
2,CB-003,751,30212,0.310,2.8,7181,0.1290,0.3458,0.7991,-166.264422,1.641149e+07
3,CB-004,811,64162,0.225,0.5,3357,0.0834,0.2625,0.7401,-32.743339,2.139575e+06
4,CB-005,712,38497,0.357,0.5,4712,0.1370,0.3701,0.9000,-349.635583,8.896243e+06


## 4. Compare the three thresholds

In [7]:
rows = []
for name, thresh in THRESHOLDS.items():
    approved = applicants[applicants["predicted_pd"] < thresh]
    total_ev  = approved["ev_usd"].sum()
    port_sd   = np.sqrt(approved["var_usd"].sum())
    rows.append({
        "threshold":         name,
        "pd_cutoff":         thresh,
        "n_approved":        len(approved),
        "approval_rate_pct": round(100 * len(approved) / len(applicants), 1),
        "total_ev_usd":      total_ev,
        "portfolio_sd":      port_sd,
        "expected_utility":  total_ev - RISK_AVERSION * port_sd,
    })

comparison = pd.DataFrame(rows).set_index("threshold")

## 5. Display the comparison table

In [8]:
comparison.style.format({
    "pd_cutoff":         "{:.0%}",
    "approval_rate_pct": "{:.1f}%",
    "total_ev_usd":      "${:,.0f}",
    "portfolio_sd":      "${:,.0f}",
    "expected_utility":  "${:,.0f}",
})

,pd_cutoff,n_approved,approval_rate_pct,total_ev_usd,portfolio_sd,expected_utility
threshold,,,,,,
Strict,15%,137,27.4%,"$298,157","$37,165","$294,440"
Moderate,25%,272,54.4%,"$489,290","$80,131","$481,277"
Permissive,40%,416,83.2%,"$452,596","$122,087","$440,388"


## 6. Identify the profit-maximizing threshold

In [9]:
best_eu = comparison["expected_utility"].idxmax()
best_ev = comparison["total_ev_usd"].idxmax()
print(f"Expected-utility winner: {best_eu}  (EU = ${comparison.loc[best_eu, 'expected_utility']:,.0f})")
print(f"Expected-value winner:   {best_ev}  (EV = ${comparison.loc[best_ev, 'total_ev_usd']:,.0f})")

Expected-utility winner: Moderate  (EU = $481,277)
Expected-value winner:   Moderate  (EV = $489,290)


**Moderate (PD < 25%) wins on both expected value and expected utility.**
Approving the 25–40% PD band (Permissive) lowers EV by ~$36K *and* raises
portfolio SD by ~$42K: those 144 applicants are harmful on both dimensions.
Their mean predicted LGD (~0.81) exceeds the break-even loss severity for their
interest rates, so they are net loss-makers. The risk-adjusted penalty
(λ × ΔSD = 0.10 × $42K = $4.2K) adds to that loss rather than offsetting it.
A fixed LGD assumption would mask this by showing Moderate and Permissive as tied.